# Powerful Attention Notebook (LLM-Ready)

This notebook builds a clean and powerful attention stack that can run on CPU and RTX 4060.

It includes all requested attention variants:
- Type 1: Bidirectional Multi-Head Self-Attention (MHA)
- Type 2: Masked Causal Self-Attention
- Type 3: Multi-Query Attention (MQA)
- Type 4: Grouped-Query Attention (GQA)
- Bonus: Cross-Attention module for encoder-decoder style setups

Architecture features:
- Pre-norm transformer blocks with RMSNorm
- SwiGLU feed-forward network
- RoPE (rotary positional encoding) option
- Weight tying and clean generation loop
- Hardware-aware profiles for CPU and RTX 4060

Data flow:
Tokenizer artifact -> Embedding artifact warm-start -> Attention model -> LM training -> Generation -> Export

In [1]:
from __future__ import annotations

import json
import math
import random
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")

Device: cpu


In [2]:
@dataclass
class TokenizerConfig:
    vocab_size: int
    min_pair_freq: int
    special_tokens: Tuple[str, ...]


class BPETokenizerRuntime:
    _word_re = re.compile(r"\s+|[^\s]+")

    def __init__(self, config: TokenizerConfig, merges_raw: List[List[int]]):
        self.config = config
        self.base_vocab_size = 256
        self.special_tokens = list(config.special_tokens)

        self.special_to_id: Dict[str, int] = {}
        self.id_to_special: Dict[int, str] = {}
        self.merges: Dict[Tuple[int, int], int] = {}
        self.merges_rank: Dict[Tuple[int, int], int] = {}
        self.token_to_bytes: Dict[int, bytes] = {
            i: bytes([i]) for i in range(self.base_vocab_size)
        }

        self._init_special_tokens()
        for rank, (a, b, new_id) in enumerate(merges_raw):
            pair = (int(a), int(b))
            merged_id = int(new_id)
            self.merges[pair] = merged_id
            self.merges_rank[pair] = rank
            self.token_to_bytes[merged_id] = self.token_to_bytes[pair[0]] + self.token_to_bytes[pair[1]]

    @classmethod
    def load(cls, path: str | Path) -> "BPETokenizerRuntime":
        payload = json.loads(Path(path).read_text(encoding="utf-8"))
        conf = payload["config"]
        config = TokenizerConfig(
            vocab_size=int(conf["vocab_size"]),
            min_pair_freq=int(conf.get("min_pair_freq", 2)),
            special_tokens=tuple(conf.get("special_tokens", ["<pad>", "<bos>", "<eos>", "<unk>"])),
        )
        return cls(config, payload["merges"])

    @property
    def vocab_size(self) -> int:
        return len(self.token_to_bytes)

    def _init_special_tokens(self) -> None:
        start = self.base_vocab_size
        for i, tok in enumerate(self.special_tokens):
            tid = start + i
            self.special_to_id[tok] = tid
            self.id_to_special[tid] = tok
            self.token_to_bytes[tid] = tok.encode("utf-8")

    def _encode_chunk(self, chunk: str) -> List[int]:
        symbols: List[int] = list(chunk.encode("utf-8"))
        if len(symbols) < 2:
            return symbols

        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                rank = self.merges_rank.get(pair)
                if rank is not None and rank < best_rank:
                    best_rank = rank
                    best_pair = pair

            if best_pair is None:
                break

            merged_token = self.merges[best_pair]
            out: List[int] = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best_pair:
                    out.append(merged_token)
                    i += 2
                else:
                    out.append(symbols[i])
                    i += 1
            symbols = out

        return symbols

    def encode(self, text: str, add_bos: bool = False, add_eos: bool = False) -> List[int]:
        if text == "":
            return []

        token_ids: List[int] = []
        if add_bos and "<bos>" in self.special_to_id:
            token_ids.append(self.special_to_id["<bos>"])

        for chunk in self._word_re.findall(text):
            token_ids.extend(self._encode_chunk(chunk))

        if add_eos and "<eos>" in self.special_to_id:
            token_ids.append(self.special_to_id["<eos>"])

        return token_ids

    def decode(self, token_ids: List[int], skip_special_tokens: bool = True) -> str:
        byte_stream = bytearray()
        for tid in token_ids:
            if skip_special_tokens and tid in self.id_to_special:
                continue
            token_bytes = self.token_to_bytes.get(int(tid), b"")
            byte_stream.extend(token_bytes)
        return bytes(byte_stream).decode("utf-8", errors="replace")

In [3]:
project_root = Path("..")
tokenizer_path = Path("bpe_tokenizer_wizard.json")
embedding_artifact_path = Path("embedding_sgns_wizard.pt")
data_path = project_root / "wizard_of_oz.txt"

if not tokenizer_path.exists():
    raise FileNotFoundError("bpe_tokenizer_wizard.json not found in Research folder.")
if not data_path.exists():
    raise FileNotFoundError("wizard_of_oz.txt not found in project root.")

tokenizer = BPETokenizerRuntime.load(tokenizer_path)
raw_text = data_path.read_text(encoding="utf-8")
token_ids = tokenizer.encode(raw_text, add_bos=True, add_eos=True)
vocab_size = tokenizer.vocab_size

pretrained_token_embedding: Optional[torch.Tensor] = None
if embedding_artifact_path.exists():
    payload = torch.load(embedding_artifact_path, map_location="cpu")
    pretrained_token_embedding = payload.get("token_embedding")
    print("Found embedding warm-start artifact:", embedding_artifact_path)
else:
    print("No embedding artifact found. Training will use random token embedding init.")

split_idx = int(0.9 * len(token_ids))
train_tokens = torch.tensor(token_ids[:split_idx], dtype=torch.long)
val_tokens = torch.tensor(token_ids[split_idx:], dtype=torch.long)

print(f"Corpus chars: {len(raw_text):,}")
print(f"Total token ids: {len(token_ids):,}")
print(f"Vocab size: {vocab_size:,}")
print(f"Train tokens: {len(train_tokens):,} | Val tokens: {len(val_tokens):,}")

Found embedding warm-start artifact: embedding_sgns_wizard.pt
Corpus chars: 232,309
Total token ids: 102,130
Vocab size: 2,000
Train tokens: 91,917 | Val tokens: 10,213


In [4]:
@dataclass
class AttentionModelConfig:
    vocab_size: int
    d_model: int
    n_layers: int
    n_heads: int
    n_kv_heads: int
    dropout: float
    max_seq_len: int
    ffn_mult: float = 3.5
    use_rope: bool = True
    tie_weights: bool = True
    attn_variant: str = "gqa"  # mha | causal_mha | mqa | gqa


@dataclass
class AttentionTrainConfig:
    batch_size: int
    lr: float
    weight_decay: float
    steps: int
    warmup_steps: int
    eval_interval: int
    eval_iters: int
    grad_accum_steps: int
    clip_grad: float
    max_new_tokens: int


def build_profiles(vocab_size: int) -> Dict[str, Tuple[AttentionModelConfig, AttentionTrainConfig]]:
    return {
        "cpu_safe": (
            AttentionModelConfig(
                vocab_size=vocab_size, d_model=192, n_layers=4, n_heads=6, n_kv_heads=6,
                dropout=0.1, max_seq_len=128, use_rope=True, attn_variant="causal_mha",
            ),
            AttentionTrainConfig(
                batch_size=24, lr=3e-4, weight_decay=0.1, steps=60, warmup_steps=10,
                eval_interval=20, eval_iters=10, grad_accum_steps=1, clip_grad=1.0, max_new_tokens=120,
            ),
        ),
        "cpu_quality": (
            AttentionModelConfig(
                vocab_size=vocab_size, d_model=256, n_layers=6, n_heads=8, n_kv_heads=4,
                dropout=0.1, max_seq_len=160, use_rope=True, attn_variant="gqa",
            ),
            AttentionTrainConfig(
                batch_size=16, lr=2.5e-4, weight_decay=0.1, steps=120, warmup_steps=20,
                eval_interval=30, eval_iters=12, grad_accum_steps=1, clip_grad=1.0, max_new_tokens=140,
            ),
        ),
        "rtx_4060_balanced": (
            AttentionModelConfig(
                vocab_size=vocab_size, d_model=512, n_layers=8, n_heads=8, n_kv_heads=4,
                dropout=0.1, max_seq_len=256, use_rope=True, attn_variant="gqa",
            ),
            AttentionTrainConfig(
                batch_size=32, lr=3e-4, weight_decay=0.1, steps=400, warmup_steps=40,
                eval_interval=50, eval_iters=20, grad_accum_steps=1, clip_grad=1.0, max_new_tokens=180,
            ),
        ),
        "rtx_4060_quality": (
            AttentionModelConfig(
                vocab_size=vocab_size, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4,
                dropout=0.1, max_seq_len=384, use_rope=True, attn_variant="gqa",
            ),
            AttentionTrainConfig(
                batch_size=20, lr=2.5e-4, weight_decay=0.1, steps=800, warmup_steps=80,
                eval_interval=80, eval_iters=24, grad_accum_steps=1, clip_grad=1.0, max_new_tokens=220,
            ),
        ),
        "rtx_4060_max": (
            AttentionModelConfig(
                vocab_size=vocab_size, d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4,
                dropout=0.1, max_seq_len=512, use_rope=True, attn_variant="gqa",
            ),
            AttentionTrainConfig(
                batch_size=10, lr=2e-4, weight_decay=0.1, steps=1200, warmup_steps=120,
                eval_interval=100, eval_iters=24, grad_accum_steps=2, clip_grad=1.0, max_new_tokens=260,
            ),
        ),
    }


profiles = build_profiles(vocab_size)
gpu_vram_gb = 0.0
if device.type == "cuda":
    gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

if device.type == "cuda" and gpu_vram_gb >= 7.5:
    profile_name = "rtx_4060_quality"
elif device.type == "cuda":
    profile_name = "rtx_4060_balanced"
else:
    profile_name = "cpu_safe"

model_cfg, train_cfg = profiles[profile_name]
print("Selected profile:", profile_name)
print("Model config:", model_cfg)
print("Train config:", train_cfg)

Selected profile: cpu_safe
Model config: AttentionModelConfig(vocab_size=2000, d_model=192, n_layers=4, n_heads=6, n_kv_heads=6, dropout=0.1, max_seq_len=128, ffn_mult=3.5, use_rope=True, tie_weights=True, attn_variant='causal_mha')
Train config: AttentionTrainConfig(batch_size=24, lr=0.0003, weight_decay=0.1, steps=60, warmup_steps=10, eval_interval=20, eval_iters=10, grad_accum_steps=1, clip_grad=1.0, max_new_tokens=120)


In [5]:
def adapt_pretrained_embedding(weight: torch.Tensor, target_dim: int, seed: int = 42) -> torch.Tensor:
    if weight.ndim != 2:
        raise ValueError("Expected embedding weight with shape [vocab_size, dim].")

    src_vocab, src_dim = weight.shape
    if src_vocab != vocab_size:
        raise ValueError(f"Embedding vocab mismatch: {src_vocab} vs tokenizer vocab {vocab_size}.")

    if src_dim == target_dim:
        return weight.float()

    if src_dim > target_dim:
        return weight[:, :target_dim].float()

    rng = torch.Generator().manual_seed(seed)
    pad = torch.randn(src_vocab, target_dim - src_dim, generator=rng) * (0.02 / math.sqrt(target_dim))
    return torch.cat([weight.float(), pad], dim=1)


def get_batch(
    split: str,
    batch_size: int,
    seq_len: int,
    device: torch.device,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
    data = train_tokens if split == "train" else val_tokens
    if len(data) <= seq_len + 1:
        raise ValueError("Sequence length is larger than available token buffer.")

    idx = torch.randint(0, len(data) - seq_len - 1, (batch_size,))
    x = torch.stack([data[i : i + seq_len] for i in idx])
    y = torch.stack([data[i + 1 : i + seq_len + 1] for i in idx])
    return x.to(device), y.to(device)


def cosine_lr(step: int, total_steps: int, warmup_steps: int, base_lr: float, min_lr_ratio: float = 0.1) -> float:
    if step < warmup_steps:
        return base_lr * (step + 1) / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return base_lr * (min_lr_ratio + (1 - min_lr_ratio) * cosine)

In [6]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.scale * x * rms


class SwiGLU(nn.Module):
    def __init__(self, dim: int, ffn_mult: float = 3.5, dropout: float = 0.0):
        super().__init__()
        hidden = int(ffn_mult * dim)
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(dim, hidden, bias=False)
        self.w_out = nn.Linear(hidden, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.silu(self.w1(x)) * self.w2(x)
        x = self.w_out(x)
        return self.dropout(x)


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, base: float = 10000.0):
        super().__init__()
        if head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE.")
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def get_cos_sin(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> Tuple[torch.Tensor, torch.Tensor]:
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos = emb.cos().to(dtype=dtype).unsqueeze(0).unsqueeze(0)
        sin = emb.sin().to(dtype=dtype).unsqueeze(0).unsqueeze(0)
        return cos, sin


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    return (x * cos) + (rotate_half(x) * sin)


class FlexibleAttention(nn.Module):
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        n_kv_heads: Optional[int] = None,
        dropout: float = 0.0,
        causal: bool = False,
        use_rope: bool = True,
    ):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads or n_heads
        self.causal = causal
        self.dropout = dropout

        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads.")
        if n_heads % self.n_kv_heads != 0:
            raise ValueError("n_heads must be divisible by n_kv_heads for grouped KV sharing.")

        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(n_heads * self.head_dim, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)

        self.rope = RotaryEmbedding(self.head_dim) if use_rope else None

    def _repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        if self.n_kv_heads == self.n_heads:
            return x
        repeat_factor = self.n_heads // self.n_kv_heads
        return x.repeat_interleave(repeat_factor, dim=1)

    def forward(
        self,
        x: torch.Tensor,
        context: Optional[torch.Tensor] = None,
        attn_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        context = x if context is None else context
        bsz, tgt_len, _ = x.shape
        src_len = context.shape[1]

        q = self.q_proj(x).view(bsz, tgt_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(context).view(bsz, src_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(context).view(bsz, src_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if self.rope is not None and context is x:
            cos_q, sin_q = self.rope.get_cos_sin(tgt_len, x.device, q.dtype)
            q = apply_rope(q, cos_q, sin_q)
            cos_k, sin_k = self.rope.get_cos_sin(src_len, x.device, k.dtype)
            k = apply_rope(k, cos_k, sin_k)

        k = self._repeat_kv(k)
        v = self._repeat_kv(v)

        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=self.causal and (attn_mask is None) and (context is x),
        )

        out = attn_out.transpose(1, 2).contiguous().view(bsz, tgt_len, self.d_model)
        return self.o_proj(self.attn_drop(out))


class MultiHeadSelfAttention(FlexibleAttention):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0, use_rope: bool = True):
        super().__init__(d_model=d_model, n_heads=n_heads, n_kv_heads=n_heads, dropout=dropout, causal=False, use_rope=use_rope)


class CausalSelfAttention(FlexibleAttention):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0, use_rope: bool = True):
        super().__init__(d_model=d_model, n_heads=n_heads, n_kv_heads=n_heads, dropout=dropout, causal=True, use_rope=use_rope)


class MultiQueryAttention(FlexibleAttention):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0, causal: bool = True, use_rope: bool = True):
        super().__init__(d_model=d_model, n_heads=n_heads, n_kv_heads=1, dropout=dropout, causal=causal, use_rope=use_rope)


class GroupedQueryAttention(FlexibleAttention):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, dropout: float = 0.0, causal: bool = True, use_rope: bool = True):
        super().__init__(d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads, dropout=dropout, causal=causal, use_rope=use_rope)


class CrossAttention(FlexibleAttention):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: Optional[int] = None, dropout: float = 0.0, use_rope: bool = False):
        super().__init__(d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads or n_heads, dropout=dropout, causal=False, use_rope=use_rope)

    def forward(self, x: torch.Tensor, context: torch.Tensor, attn_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        return super().forward(x=x, context=context, attn_mask=attn_mask)

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg: AttentionModelConfig):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.norm2 = RMSNorm(cfg.d_model)

        if cfg.attn_variant == "mha":
            self.attn = MultiHeadSelfAttention(cfg.d_model, cfg.n_heads, dropout=cfg.dropout, use_rope=cfg.use_rope)
        elif cfg.attn_variant == "causal_mha":
            self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, dropout=cfg.dropout, use_rope=cfg.use_rope)
        elif cfg.attn_variant == "mqa":
            self.attn = MultiQueryAttention(cfg.d_model, cfg.n_heads, dropout=cfg.dropout, causal=True, use_rope=cfg.use_rope)
        elif cfg.attn_variant == "gqa":
            self.attn = GroupedQueryAttention(
                cfg.d_model, cfg.n_heads, cfg.n_kv_heads, dropout=cfg.dropout, causal=True, use_rope=cfg.use_rope
            )
        else:
            raise ValueError(f"Unknown attention variant: {cfg.attn_variant}")

        self.ffn = SwiGLU(cfg.d_model, ffn_mult=cfg.ffn_mult, dropout=cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class PowerfulAttentionLM(nn.Module):
    def __init__(self, cfg: AttentionModelConfig, pretrained_embedding: Optional[torch.Tensor] = None):
        super().__init__()
        self.cfg = cfg
        self.token_embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        if cfg.tie_weights:
            self.lm_head.weight = self.token_embed.weight

        self.pos_embed = None if cfg.use_rope else nn.Embedding(cfg.max_seq_len, cfg.d_model)

        if pretrained_embedding is not None:
            init_weight = adapt_pretrained_embedding(pretrained_embedding, cfg.d_model)
            self.token_embed.weight.data.copy_(init_weight)

    def forward(self, idx: torch.Tensor, targets: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        bsz, seq_len = idx.shape
        if seq_len > self.cfg.max_seq_len:
            raise ValueError(f"Sequence length {seq_len} exceeds max_seq_len {self.cfg.max_seq_len}.")

        x = self.token_embed(idx)
        if self.pos_embed is not None:
            pos = torch.arange(seq_len, device=idx.device).unsqueeze(0).expand(bsz, seq_len)
            x = x + self.pos_embed(pos)

        x = self.dropout(x)
        for block in self.blocks:
            x = block(x)

        x = self.norm_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: Optional[int] = 50,
    ) -> torch.Tensor:
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.max_seq_len :]
            logits, _ = self(idx_cond)
            next_logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None:
                top_vals, top_idx = torch.topk(next_logits, k=min(top_k, next_logits.size(-1)), dim=-1)
                filtered = torch.full_like(next_logits, float("-inf"))
                filtered.scatter_(1, top_idx, top_vals)
                next_logits = filtered

            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


@torch.no_grad()
def estimate_loss(model: nn.Module, cfg: AttentionTrainConfig, model_cfg: AttentionModelConfig, eval_iters: int) -> Dict[str, float]:
    out = {}
    model.eval()
    for split in ("train", "val"):
        losses = []
        for _ in range(eval_iters):
            xb, yb = get_batch(split, cfg.batch_size, model_cfg.max_seq_len, device)
            _, loss = model(xb, yb)
            losses.append(float(loss.item()))
        out[split] = float(np.mean(losses))
    model.train()
    return out

In [8]:
# Smoke test all requested attention types
with torch.no_grad():
    x = torch.randn(2, 64, model_cfg.d_model, device=device)
    context = torch.randn(2, 80, model_cfg.d_model, device=device)

    attn_type_1 = MultiHeadSelfAttention(model_cfg.d_model, model_cfg.n_heads, dropout=0.0, use_rope=model_cfg.use_rope).to(device)
    out_1 = attn_type_1(x)

    attn_type_2 = CausalSelfAttention(model_cfg.d_model, model_cfg.n_heads, dropout=0.0, use_rope=model_cfg.use_rope).to(device)
    out_2 = attn_type_2(x)

    attn_type_3 = MultiQueryAttention(model_cfg.d_model, model_cfg.n_heads, dropout=0.0, causal=True, use_rope=model_cfg.use_rope).to(device)
    out_3 = attn_type_3(x)

    kv_heads = max(1, model_cfg.n_heads // 4)
    if model_cfg.n_heads % kv_heads != 0:
        kv_heads = 1
    attn_type_4 = GroupedQueryAttention(
        model_cfg.d_model, model_cfg.n_heads, n_kv_heads=kv_heads, dropout=0.0, causal=True, use_rope=model_cfg.use_rope
    ).to(device)
    out_4 = attn_type_4(x)

    cross_attn = CrossAttention(model_cfg.d_model, model_cfg.n_heads, n_kv_heads=kv_heads, dropout=0.0, use_rope=False).to(device)
    out_cross = cross_attn(x, context=context)

print("Type 1 MHA output:", tuple(out_1.shape))
print("Type 2 Masked Causal output:", tuple(out_2.shape))
print("Type 3 MQA output:", tuple(out_3.shape))
print("Type 4 GQA output:", tuple(out_4.shape))
print("Bonus Cross-Attention output:", tuple(out_cross.shape))

Type 1 MHA output: (2, 64, 192)
Type 2 Masked Causal output: (2, 64, 192)
Type 3 MQA output: (2, 64, 192)
Type 4 GQA output: (2, 64, 192)
Bonus Cross-Attention output: (2, 64, 192)


In [9]:
model = PowerfulAttentionLM(model_cfg, pretrained_embedding=pretrained_token_embedding).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {num_params / 1_000_000:.2f}M")

history = {"step": [], "train_loss": [], "val_loss": []}
start_time = time.perf_counter()
model.train()
optimizer.zero_grad(set_to_none=True)

for step in range(train_cfg.steps):
    lr = cosine_lr(step, train_cfg.steps, train_cfg.warmup_steps, train_cfg.lr)
    for group in optimizer.param_groups:
        group["lr"] = lr

    if step % train_cfg.eval_interval == 0 or step == train_cfg.steps - 1:
        losses = estimate_loss(model, train_cfg, model_cfg, eval_iters=train_cfg.eval_iters)
        history["step"].append(step)
        history["train_loss"].append(losses["train"])
        history["val_loss"].append(losses["val"])
        print(
            f"step={step:4d} | lr={lr:.6f} | train_loss={losses['train']:.4f} | val_loss={losses['val']:.4f}"
        )

    xb, yb = get_batch("train", train_cfg.batch_size, model_cfg.max_seq_len, device)

    if device.type == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            _, loss = model(xb, yb)
            loss = loss / train_cfg.grad_accum_steps
        scaler.scale(loss).backward()
    else:
        _, loss = model(xb, yb)
        loss = loss / train_cfg.grad_accum_steps
        loss.backward()

    should_step = ((step + 1) % train_cfg.grad_accum_steps == 0) or (step == train_cfg.steps - 1)
    if should_step:
        if train_cfg.clip_grad > 0:
            if device.type == "cuda":
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.clip_grad)

        if device.type == "cuda":
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

elapsed = time.perf_counter() - start_time
print(f"Training completed in {elapsed:.2f}s")

Trainable parameters: 2.52M
step=   0 | lr=0.000030 | train_loss=8.1264 | val_loss=8.1273
step=  20 | lr=0.000274 | train_loss=4.7140 | val_loss=4.7797
step=  40 | lr=0.000123 | train_loss=4.2262 | val_loss=4.3147
step=  59 | lr=0.000030 | train_loss=4.1344 | val_loss=4.2596
Training completed in 95.90s


In [10]:
prompt = "Dorothy "
prompt_ids = tokenizer.encode(prompt, add_bos=True, add_eos=False)
prompt_tensor = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)

with torch.no_grad():
    generated = model.generate(
        prompt_tensor,
        max_new_tokens=train_cfg.max_new_tokens,
        temperature=0.9,
        top_k=60,
    )

generated_text = tokenizer.decode(generated[0].tolist(), skip_special_tokens=True)
print("Prompt:", repr(prompt))
print("Generated sample:")
print(generated_text[:1200])

artifact_path = Path("attention_model_wizard.pt")
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_config": model_cfg.__dict__,
        "train_config": train_cfg.__dict__,
        "profile": profile_name,
        "history": history,
        "tokenizer_json": str(tokenizer_path),
        "embedding_artifact": str(embedding_artifact_path) if embedding_artifact_path.exists() else None,
    },
    artifact_path,
)
print("Saved attention artifact:", artifact_path.resolve())

Prompt: 'Dorothy '
Generated sample:
Dorothy her much of they no hadG

mly.



will of and we the is if a in had you who in but in went be there the in to and
of his the went out some and the were so that but for for they as the and the they said a on."

you of of was a that 
Saved attention artifact: D:\Desktop\Mini_Generative_Pretrained_Transformer\Research\attention_model_wizard.pt


## Notes for next scaling stage

- To scale this model with RTX 4060, switch profile to `rtx_4060_balanced` or `rtx_4060_quality`.
- For longer context and better extrapolation, keep RoPE enabled and increase `max_seq_len` gradually.
- For memory issues, reduce in this order: batch size -> sequence length -> number of layers -> d_model.
- For best final quality, continue training with your larger dataset and then move to full multi-block architecture notebook.